In [1]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

c:\Users\Menna\AppData\Local\Programs\Python\Python38\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
import sys
sys.path.append('../../../')
from clean_all import clean
clean()
sys.path.pop()

'../../../'

In [3]:
config = {
    "lib": "tensorflow",
    "mode": 'local',
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 2,
    "batch_size": 128,
    "loss": tf.keras.losses.CategoricalCrossentropy(),
    "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )


def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)

    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size :])
            y_train_partitions.append(y_train[i * partition_size :])
        else:
            X_train_partitions.append(
                X_train[i * partition_size : (i + 1) * partition_size]
            )
            y_train_partitions.append(
                y_train[i * partition_size : (i + 1) * partition_size]
            )

    return X_train_partitions, y_train_partitions

In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()
X_train, y_train = partition_train_data(X_train, y_train, config["partitions"])

In [7]:
for i in range(len(X_train)):
    np.save(f"../../../data/X_train_{i + 1}.npy", X_train[i])
    np.save(f"../../../data/y_train_{i + 1}.npy", y_train[i])

In [8]:
model = create_model()
rain = Rain(config, model, X_train, y_train)

2023-07-02 17:23:11,907 [DEBUG] [Rain] Rain is initialized
2023-07-02 17:23:11,909 [DEBUG] [Provisioner] Creating coordinator
2023-07-02 17:23:11,911 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-02 17:23:11,913 [DEBUG] [LocalProvisioner] LocalProvisioner is initialized


In [9]:
model = rain.train_centralized_async()

2023-07-02 17:23:12,048 [DEBUG] [Rain] Creating workers
2023-07-02 17:23:12,062 [INFO] [Provisioner] provisioner is serving
2023-07-02 17:23:12,064 [DEBUG] [Provisioner] Starting coordinator
2023-07-02 17:23:12,081 [INFO] [Coordinator] coordinator is serving
2023-07-02 17:23:12,083 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-02 17:23:12,094 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-02 17:23:12,106 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-02 17:23:12,108 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-02 17:23:12,128 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-02 17:23:12,132 [INFO] [Worker_50152] Worker is running on port: 50152
2023-07-02 17:23:12,137 [INFO] [Worker_50153] Worker is running on port: 50153
2023-07-02 17:23:12,138 [DEBUG] [Provisioner] [Created workers]
IPs : ['127.0.0.1', '127.0.0.1', '127.0.0

In [10]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 3ms/step - loss: 0.1023 - accuracy: 0.9699

Test accuracy: 97.0%


In [11]:
model = rain.train_centralized_sync()

2023-07-02 17:26:42,140 [DEBUG] [Rain] Creating workers
2023-07-02 17:26:42,144 [INFO] [Provisioner] provisioner is serving
2023-07-02 17:26:42,146 [DEBUG] [Provisioner] Starting coordinator
2023-07-02 17:26:42,150 [ERROR] [Coordinator] Error in the coordinator server: Failed to bind to address [::]:50052; set GRPC_VERBOSITY=debug environment variable to see detailed error message.
2023-07-02 17:26:42,152 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-02 17:26:42,155 [ERROR] [Worker_50151] Error in the worker server: Failed to bind to address [::]:50151; set GRPC_VERBOSITY=debug environment variable to see detailed error message.
--- Logging error ---
Traceback (most recent call last):
  File "../../..\Worker\Worker.py", line 89, in serve
    self.server.add_insecure_port(f'[::]:{self.port}')
  File "c:\Users\Menna\AppData\Local\Programs\Python\Python38\lib\site-packages\grpc\_server.py", line 1101, in add_insecure_port
    return _common.validate_port_binding_result(
  File "c:

In [12]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

AttributeError: 'NoneType' object has no attribute 'evaluate'